# Part 3: Adversarial robustness

**Attack 1:** character-level evasion on high-confidence toxic predictions. **Attack 2:** 5% random label flips, retrain DistilBERT from pretrained, compare to the Part 1 checkpoint at the same threshold.


In [ ]:
import subprocess, sys

_pkgs = ["transformers", "datasets", "accelerate", "torch", "scikit-learn", "pandas", "numpy"]
for p in _pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])
print("Dependencies OK.")


All packages ready.


In [ ]:
import os, random, warnings

warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

CHECKPOINT_DIR = "/content/drive/My Drive/Colab Notebooks/distilbert_toxic_checkpoint"
POISONED_CKPT_DIR = "/content/distilbert_poisoned_checkpoint"
DATA_PATH = "jigsaw-unintended-bias-train.csv"
THRESHOLD = 0.40
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
BATCH_SIZE = 32
EPOCHS = 3
TRAIN_SIZE, EVAL_SIZE = 100_000, 20_000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"{DEVICE=}")


Device : cpu


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

prob_toxic = np.load(os.path.join(CHECKPOINT_DIR, "eval_probs.npy"))
true_labels = np.load(os.path.join(CHECKPOINT_DIR, "eval_labels.npy"))
df_eval = pd.read_parquet(os.path.join(CHECKPOINT_DIR, "eval_df.parquet"))
df_eval["prob_toxic"] = prob_toxic
df_eval["pred_label"] = (prob_toxic >= THRESHOLD).astype(int)
df_eval["true_label"] = true_labels.astype(int)
print(len(df_eval), list(df_eval.columns))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Eval set loaded : 20,000 rows
Columns         : ['comment_text', 'toxic', 'black', 'jewish', 'muslim', 'other_sexual_orientation', 'white', 'label', 'prob_toxic', 'pred_label', 'true_label']


## Attack 1: Character-level evasion


In [ ]:
HOM = {"a": "а", "e": "е", "o": "о", "p": "р", "c": "с", "x": "х"}
ZWS = "​"


def insert_zws(word: str) -> str:
    out, i = [], 0
    while i < len(word):
        step = random.choice([2, 3])
        out.append(word[i : i + step])
        if i + step < len(word):
            out.append(ZWS)
        i += step
    return "".join(out)


def homoglyphs(word: str) -> str:
    return "".join(HOM.get(ch, ch) for ch in word)


def dup_chars(word: str, rate: float = 0.2) -> str:
    out = []
    for ch in word:
        out.append(ch)
        if random.random() < rate:
            out.append(ch)
    return "".join(out)


def perturb(text: str) -> str:
    return " ".join(dup_chars(homoglyphs(insert_zws(w))) for w in text.split())


sample = "I hate you so much"
print(sample)
print(perturb(sample))


Original  : I hate you so much
Perturbed : II hаt​ее yyооu sо muс​hh
(Perturbed text looks similar to human but is different to tokenizer)


In [ ]:
pool = df_eval[(df_eval["pred_label"] == 1) & (df_eval["prob_toxic"] >= 0.7)]
n = min(500, len(pool))
df_attack1 = pool.sample(n=n, random_state=SEED).reset_index(drop=True)
df_attack1["perturbed_text"] = df_attack1["comment_text"].map(perturb)
print(n, df_attack1["comment_text"].iloc[0][:80], "→", df_attack1["perturbed_text"].iloc[0][:80])


High-confidence toxic comments available : 1,095
Applying perturb() to 500 comments ...
Done. Sample comparison:
  Original  : The Islamic State's comments apply equally to Canada. They won't criticize Littl
  Perturbed : Thhе Isl​ааm​​iсс Sttа​ttе''​s соm​mmе​nts ар​ррly еq​uаl​ly tо Cа​​nnаd​а. Thhе


In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained(CHECKPOINT_DIR)
clean_model = DistilBertForSequenceClassification.from_pretrained(CHECKPOINT_DIR).to(DEVICE)
clean_model.eval()


def get_probs(texts, model, batch_size=64):
    out = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(
            texts[i : i + batch_size],
            max_length=MAX_LENGTH,
            truncation=True,
            padding=True,
            return_tensors="pt",
        ).to(DEVICE)
        with torch.no_grad():
            logits = model(**enc).logits
        out.extend(F.softmax(logits, -1)[:, 1].cpu().numpy())
    return np.array(out)


df_attack1["orig_prob"] = get_probs(df_attack1["comment_text"].tolist(), clean_model)
df_attack1["pert_prob"] = get_probs(df_attack1["perturbed_text"].tolist(), clean_model)
df_attack1["pert_label"] = (df_attack1["pert_prob"] >= THRESHOLD).astype(int)
print("scored", len(df_attack1))


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Scoring original comments  ...
Scoring perturbed comments ...
Inference complete.


In [ ]:
asr = (df_attack1["pert_label"] == 0).mean()
drop = df_attack1["orig_prob"].mean() - df_attack1["pert_prob"].mean()
print(f"ASR={asr:.4f}  Δconf={drop:.4f}  n={len(df_attack1)}")
pd.DataFrame(
    [
        {"metric": "attack_success_rate", "value": f"{asr:.4f}"},
        {"metric": "mean_conf_before", "value": f"{df_attack1['orig_prob'].mean():.4f}"},
        {"metric": "mean_conf_after", "value": f"{df_attack1['pert_prob'].mean():.4f}"},
    ]
)


  ATTACK 1: CHARACTER-LEVEL EVASION RESULTS
  Comments sampled          : 500
  Successfully evaded       : 498
  Attack Success Rate (ASR) : 0.9960  (99.6%)
  Avg confidence BEFORE     : 0.9094
  Avg confidence AFTER      : 0.0085
  Confidence drop           : 0.9009
               Metric          Value
  Attack Success Rate 0.9960 (99.6%)
Avg Confidence Before         0.9094
 Avg Confidence After         0.0085
      Confidence Drop         0.9009


## Attack 2: Label-flip poisoning (5%)


In [ ]:
df_full = pd.read_csv(DATA_PATH, usecols=["comment_text", "toxic"], low_memory=False, encoding="latin1")
df_full["toxic"] = pd.to_numeric(df_full["toxic"].astype(str), errors="coerce")
df_full = df_full.dropna(subset=["comment_text", "toxic"])
df_full["comment_text"] = df_full["comment_text"].astype(str)
df_full["label"] = (df_full["toxic"] >= 0.5).astype(int)
sub, _ = train_test_split(df_full, train_size=TRAIN_SIZE + EVAL_SIZE, stratify=df_full["label"], random_state=SEED)
df_train, _ = train_test_split(sub, train_size=TRAIN_SIZE, stratify=sub["label"], random_state=SEED)
df_train = df_train.reset_index(drop=True)
print(len(df_train), df_train["label"].value_counts().to_dict())


Loading jigsaw-unintended-bias-train.csv ...
Training subset reconstructed : 100,000 rows
Label distribution:
label
0    91917
1     8083
Name: count, dtype: int64


In [ ]:
n_poison = int(len(df_train) * 0.05)
idx = np.random.RandomState(SEED).choice(df_train.index, size=n_poison, replace=False)
df_poisoned = df_train.copy()
df_poisoned.loc[idx, "label"] = 1 - df_poisoned.loc[idx, "label"]
print(f"flipped {n_poison} labels ({100 * n_poison / len(df_train):.1f}%)")


Rows poisoned (label flipped) : 5,000  (5%)

Label distribution BEFORE poisoning:
label
0    91917
1     8083
Name: count, dtype: int64

Label distribution AFTER poisoning:
label
0    87683
1    12317
Name: count, dtype: int64


In [ ]:
def tokenize_fn(batch):
    return tokenizer(batch["comment_text"], max_length=MAX_LENGTH, truncation=True, padding="max_length")


eval_hf = Dataset.from_pandas(df_eval[["comment_text", "true_label"]].rename(columns={"true_label": "label"}))
eval_tok = eval_hf.map(tokenize_fn, batched=True, batch_size=1000, remove_columns=["comment_text"])
eval_tok = eval_tok.rename_column("label", "labels")
eval_tok.set_format("torch")

train_poison_tok = (
    Dataset.from_pandas(df_poisoned[["comment_text", "label"]])
    .map(tokenize_fn, batched=True, batch_size=1000, remove_columns=["comment_text"])
    .rename_column("label", "labels")
)
train_poison_tok.set_format("torch")


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Tokenizing poisoned training set ...


Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Tokenization complete.


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    pred = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, pred), "f1_macro": f1_score(labels, pred, average="macro")}


poisoned_model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
args = TrainingArguments(
    output_dir=POISONED_CKPT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_ratio=0.1,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=200,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    report_to="none",
)
trainer_poisoned = Trainer(
    model=poisoned_model,
    args=args,
    train_dataset=train_poison_tok,
    eval_dataset=eval_tok,
    compute_metrics=compute_metrics,
)
trainer_poisoned.train()


In [ ]:
log_p = trainer_poisoned.predict(eval_tok).predictions
prob_p = F.softmax(torch.tensor(log_p), dim=-1).numpy()[:, 1]
pred_p = (prob_p >= THRESHOLD).astype(int)
pred_c = (prob_toxic >= THRESHOLD).astype(int)


def fnr_fpr(y_t, y_p):
    tn, fp, fn, tp = confusion_matrix(y_t, y_p, labels=[0, 1]).ravel()
    return (fn / (fn + tp) if (fn + tp) else 0.0, fp / (fp + tn) if (fp + tn) else 0.0)


fnr_c, fpr_c = fnr_fpr(true_labels, pred_c)
fnr_p, fpr_p = fnr_fpr(true_labels, pred_p)
comparison = pd.DataFrame(
    [
        {
            "model": "clean",
            "acc": round(accuracy_score(true_labels, pred_c), 4),
            "f1_macro": round(f1_score(true_labels, pred_c, average="macro"), 4),
            "fnr": round(fnr_c, 4),
            "fpr": round(fpr_c, 4),
        },
        {
            "model": "poisoned",
            "acc": round(accuracy_score(true_labels, pred_p), 4),
            "f1_macro": round(f1_score(true_labels, pred_p, average="macro"), 4),
            "fnr": round(fnr_p, 4),
            "fpr": round(fpr_p, 4),
        },
        {
            "model": "delta",
            "acc": round(accuracy_score(true_labels, pred_p) - accuracy_score(true_labels, pred_c), 4),
            "f1_macro": round(f1_score(true_labels, pred_p, average="macro") - f1_score(true_labels, pred_c, average="macro"), 4),
            "fnr": round(fnr_p - fnr_c, 4),
            "fpr": round(fpr_p - fpr_c, 4),
        },
    ]
)
print(comparison.to_string(index=False))


## Which attack is more dangerous?

**Evasion** is cheap at scale for an attacker who only controls text at inference time; **poisoning** needs data-pipeline access but can shift decisions for every user. Compare ASR (Attack 1) to FNR / F1 shifts (Attack 2) when prioritizing mitigations.
